# Plot Light Curves vs MJD for Stable Stars — Comparison with Calibration Parameters

This notebook builds on `03_PlotLCwithMJD.ipynb` (which plots `psfFlux` vs
`expMidptMJD` per band, with a median +/- sigma_IQR envelope). Here the goal
is different: **for each stable star, compare the psfFlux light curve with
the calibration / atmospheric quantities that may explain outliers**:

- `calib_local`   — local PhotoCalib calibration coefficient (notebook 05)
- `zeropoint`      — PhotoCalib zero-point [AB mag] (notebook 05)
- FGCM atmospheric parameter relevant to the band (notebook 08):
    - **u, g bands** -> `fgcm_tau`  (aerosol optical depth)
    - **r, i bands** -> `fgcm_o3`   (ozone column)
    - **z, y bands** -> `fgcm_pwv`  (precipitable water vapour)
- `airmass`        — observation airmass (all bands)

For each star, a single **large figure (one page per star)** is produced:
a GridSpec with **6 columns (bands u g r i z y)** and **5 rows**
(psfFlux, calib_local, zeropoint, FGCM atm. parameter, airmass), all
sharing the same MJD x-axis within a column.

The `psfFlux` row reproduces the median +/- sigma_IQR envelope of
`03_PlotLCwithMJD.ipynb`. Points deviating from the median by more than
`OUTLIER_NSIGMA * sigma_IQR` are flagged as outliers and highlighted with a
red star marker **in every row of that band's column**, making it possible
to see at a glance whether psfFlux outliers coincide with excursions in the
local calibration, zero-point, atmospheric conditions, or airmass.

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-07-01
- **Last update:** 2026-07-01


## 1. Imports

In [ ]:
import gc
import logging
import os
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator

from astropy.time import Time

In [ ]:
# Show all rows/columns when displaying pandas tables
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [ ]:
# Enable interactive matplotlib backend if ipympl is available
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found -> interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found -> %matplotlib inline")

## 2. Logging

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)
if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)
log.info("Logging initialised.")

## 3. Configuration

In [ ]:
# ── Notebook tag ──────────────────────────────────────────────────────
NB_TAG = "PlotLCCompareCalibs_10"

# ── Input: FGCM-enriched per-star light curves (output of notebook 08) ──
DIR_DATA_IN = "./data_AddFGCM_08_out"
DIR_PER_STAR_IN = os.path.join(DIR_DATA_IN, "per_star")
PER_STAR_SUFFIX = "_lc_fgcm.csv"
GLOBAL_FGCM_FILE = "all_stars_lightcurves_fgcm.csv"

# ── Output figures ────────────────────────────────────────────────────
DIR_FIGS = f"./figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
log.info("Figure directory: %s", DIR_FIGS)

# ── Photometric / calibration columns ───────────────────────────────────
MJD_COL = "expMidptMJD"
FLUX_COL = "psfFlux"
FLUX_ERR_COL = "psfFluxErr"
BAND_COL = "band"
CALIB_LOCAL_COL = "calib_local"
ZEROPOINT_COL = "zeropoint"
AIRMASS_COL = "airmass"

# ── Band ordering and colours (LSST ugrizy) ──────────────────────────────
BAND_ORDER = ["u", "g", "r", "i", "z", "y"]
BAND_COLORS = {
    "u": "#6C2DC7",  # violet
    "g": "#1CB94A",  # green
    "r": "#E8262A",  # red
    "i": "#E87E26",  # orange
    "z": "#C95C93",  # pink
    "y": "#994D00",  # brown
}

# ── Colours for the calibration / atmospheric rows (fixed per row type) ──
ROW_COLORS = {
    "calib_local": "purple",
    "zeropoint": "royalblue",
    "atm": "seagreen",
    "airmass": "dimgray",
}

# ── FGCM atmospheric parameter relevant to each band ─────────────────────
# u, g   -> aerosol optical depth (tau)
# r, i   -> ozone column (O3)
# z, y   -> precipitable water vapour (PWV)
ATM_PARAM_BY_BAND = {
    "u": ("fgcm_tau", "FGCM aerosol tau"),
    "g": ("fgcm_tau", "FGCM aerosol tau"),
    "r": ("fgcm_o3", "FGCM ozone O3 [DU]"),
    "i": ("fgcm_o3", "FGCM ozone O3 [DU]"),
    "z": ("fgcm_pwv", "FGCM PWV [mm]"),
    "y": ("fgcm_pwv", "FGCM PWV [mm]"),
}

# ── Row layout of the per-star figure (top to bottom) ────────────────────
ROW_KEYS = ["psfFlux", "calib_local", "zeropoint", "atm", "airmass"]

# ── Y-axis clipping for the psfFlux row ───────────────────────────────────
# median +/- N_SIGMA_YLIM * sigma_IQR   (sigma_IQR = IQR / 1.349)
N_SIGMA_YLIM = 5.0

# ── Outlier threshold: points beyond this many sigma_IQR from the median
# are highlighted (red star) in every row of the corresponding band column.
OUTLIER_NSIGMA = 3.0

# ── Minimum number of good points to plot a band column ──────────────────
MIN_POINTS = 5

# ── Figure size (width, height) in inches for the 5x6 per-star figure ────
FIG_WIDTH = 18
FIG_HEIGHT = 10

## 4. Helper functions

In [ ]:
# ── Robust sigma from the inter-quartile range ────────────────────────
def sigma_iqr(values):
    """Robust standard-deviation estimate via the inter-quartile range.
    sigma_IQR = IQR / 1.3489795
    """
    q75, q25 = np.nanpercentile(values, [75, 25])
    return (q75 - q25) / 1.3489795


# ── savefig: PDF + PNG ───────────────────────────────────────────────
def savefig(fig, name, dpi=150):
    """Save *fig* as both PDF and PNG under DIR_FIGS."""
    base = os.path.join(DIR_FIGS, name)
    fig.savefig(base + ".pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(base + ".png", dpi=dpi, bbox_inches="tight")
    log.info("Saved figure: %s (.pdf/.png)", base)


# ── MJD <-> matplotlib date number (linear day-count shift) ──────────────
# Lets us add a calendar-date secondary axis on top of an MJD axis without
# a nonlinear/categorical transform.
MJD_TO_MPL_OFFSET = mdates.date2num(Time(0.0, format="mjd").to_datetime())


def _mjd_to_mpl(mjd):
    return np.asarray(mjd, dtype=float) + MJD_TO_MPL_OFFSET


def _mpl_to_mjd(mpl):
    return np.asarray(mpl, dtype=float) - MJD_TO_MPL_OFFSET


def add_date_top_axis(ax):
    """Add a secondary top x-axis showing calendar dates (YYYY-MM-DD)."""
    secax = ax.secondary_xaxis("top", functions=(_mjd_to_mpl, _mpl_to_mjd))
    secax.xaxis.set_major_locator(mdates.AutoDateLocator())
    secax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    plt.setp(secax.get_xticklabels(), rotation=30, ha="left", fontsize=6)
    return secax

## 5. Core plotting function: one star -> one 5x6 GridSpec figure

Columns = bands (`u g r i z y`); rows = `psfFlux`, `calib_local`,
`zeropoint`, FGCM atmospheric parameter (band-dependent), `airmass`.
All panels in a column share the same MJD x-axis.


In [ ]:
def plot_star_calib_comparison(df_star: pd.DataFrame, simbad_id: str, file_stem: str) -> None:
    """Build and save the 5x6 GridSpec figure comparing psfFlux with
    calibration / atmospheric quantities, for one stable star.

    Parameters
    ----------
    df_star    : per-star FGCM-enriched light-curve DataFrame.
    simbad_id  : human-readable identifier shown in the figure title.
    file_stem  : output filename stem (no extension, no directory).
    """
    df = df_star.dropna(subset=[MJD_COL, FLUX_COL]).copy()
    ra = df["ra"].mean() if "ra" in df.columns else np.nan
    dec = df["dec"].mean() if "dec" in df.columns else np.nan

    fig = plt.figure(figsize=(FIG_WIDTH, FIG_HEIGHT))
    fig.suptitle(
        f"{simbad_id} | (ra,dec)=({ra:.2f},{dec:.2f})\n"
        f"psfFlux vs. calibration / atmospheric parameters per band   |   "
        f"outliers: |flux - median| > {OUTLIER_NSIGMA:.0f}*sigma_IQR",
        fontsize=14,
        fontweight="bold",
        y=0.998,
    )

    n_rows = len(ROW_KEYS)
    n_cols = len(BAND_ORDER)
    gs = gridspec.GridSpec(n_rows, n_cols, figure=fig, hspace=0.15, wspace=0.35)

    row_label = {
        "psfFlux": "psfFlux [nJy]",
        "calib_local": "calib_local",
        "zeropoint": "zeropoint [mag]",
        "atm": "FGCM atm. param.",
        "airmass": "airmass",
    }

    for col_idx, band in enumerate(BAND_ORDER):
        band_color = BAND_COLORS.get(band, "steelblue")
        atm_col, atm_label = ATM_PARAM_BY_BAND[band]

        # Select this band, require valid flux and error
        mask = (df[BAND_COL] == band) & df[MJD_COL].notna() & df[FLUX_COL].notna()
        if FLUX_ERR_COL in df.columns:
            mask &= df[FLUX_ERR_COL].notna() & (df[FLUX_ERR_COL] > 0)
        sub = df[mask].sort_values(MJD_COL)
        n_pts = len(sub)

        # ── Empty band column ─────────────────────────────────────────
        if n_pts < MIN_POINTS:
            ax_empty = fig.add_subplot(gs[:, col_idx])
            ax_empty.text(
                0.5,
                0.5,
                f"{band} band\n{n_pts} point(s)",
                ha="center",
                va="center",
                transform=ax_empty.transAxes,
                color="grey",
                fontsize=11,
            )
            ax_empty.set_xticks([])
            ax_empty.set_yticks([])
            ax_empty.set_title(f"{band} band", fontsize=11, color=band_color, fontweight="bold")
            continue

        mjd = sub[MJD_COL].values
        flux = sub[FLUX_COL].values
        err = sub[FLUX_ERR_COL].values if FLUX_ERR_COL in sub.columns else np.zeros_like(flux)

        # ── Robust psfFlux statistics (median + sigma_IQR envelope) ────
        med = np.nanmedian(flux)
        sig_iqr = sigma_iqr(flux)
        if sig_iqr == 0:
            sig_iqr = np.nanstd(flux) or 1.0

        # ── Outlier mask, shared across all rows of this band column ───
        outlier_mask = np.abs(flux - med) > OUTLIER_NSIGMA * sig_iqr
        n_outliers = int(outlier_mask.sum())

        axes_col = []

        # ═══ Row 0: psfFlux with median +/- sigma_IQR envelope ═════════
        ax0 = fig.add_subplot(gs[0, col_idx])
        axes_col.append(ax0)

        ax0.errorbar(
            mjd,
            flux,
            yerr=err,
            fmt="o",
            ms=3.5,
            color=band_color,
            ecolor=band_color,
            alpha=0.75,
            elinewidth=0.7,
            capsize=1.5,
            zorder=2,
            label=f"N={n_pts}",
        )
        ax0.axhline(med, color=band_color, lw=1.3, ls="--", alpha=0.9, zorder=3)
        ax0.axhspan(med - sig_iqr, med + sig_iqr, color=band_color, alpha=0.12, zorder=1)

        if n_outliers > 0:
            ax0.scatter(
                mjd[outlier_mask],
                flux[outlier_mask],
                marker="*",
                s=90,
                facecolor="red",
                edgecolor="k",
                linewidth=0.6,
                zorder=5,
                label=f"outliers ({n_outliers})",
            )

        ylim_lo = med - N_SIGMA_YLIM * sig_iqr
        ylim_hi = med + N_SIGMA_YLIM * sig_iqr
        if ylim_hi - ylim_lo < 1e-10 * abs(med) + 1.0:
            ylim_lo, ylim_hi = med - 1.0, med + 1.0
        ax0.set_ylim(ylim_lo, ylim_hi)

        rel_scatter = 100.0 * sig_iqr / abs(med) if abs(med) > 0 else np.nan
        ax0.text(
            0.02,
            0.94,
            f"sigma_IQR={sig_iqr:.1f} nJy ({rel_scatter:.2f}%)",
            transform=ax0.transAxes,
            fontsize=6.5,
            va="top",
            ha="left",
            color=band_color,
        )
        ax0.legend(loc="upper right", fontsize=6, framealpha=0.6)
        ax0.set_title(f"{band} band  ({n_pts} pts)", fontsize=11, color=band_color, fontweight="bold")
        add_date_top_axis(ax0)

        # ═══ Rows 1..4: calib_local / zeropoint / atm / airmass ════════
        row_specs = [
            ("calib_local", CALIB_LOCAL_COL, row_label["calib_local"]),
            ("zeropoint", ZEROPOINT_COL, row_label["zeropoint"]),
            ("atm", atm_col, f"{row_label['atm']}\n({atm_label})"),
            ("airmass", AIRMASS_COL, row_label["airmass"]),
        ]

        for row_idx, (row_key, col_name, ylabel) in enumerate(row_specs, start=1):
            ax = fig.add_subplot(gs[row_idx, col_idx], sharex=ax0)
            axes_col.append(ax)
            color = ROW_COLORS[row_key]

            if col_name not in sub.columns:
                ax.text(
                    0.5,
                    0.5,
                    f"'{col_name}' not available",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                    color="grey",
                    fontsize=7,
                )
            else:
                vals = sub[col_name].values
                valid = np.isfinite(vals)
                if valid.sum() > 0:
                    ax.scatter(
                        mjd[valid],
                        vals[valid],
                        s=10,
                        color=color,
                        alpha=0.7,
                        marker="o",
                        zorder=2,
                    )
                    med_val = np.nanmedian(vals[valid])
                    ax.axhline(med_val, color=color, lw=1.0, ls="--", alpha=0.7, zorder=1)

                    # Highlight the same points flagged as psfFlux outliers
                    outl_here = outlier_mask & valid
                    if outl_here.any():
                        ax.scatter(
                            mjd[outl_here],
                            vals[outl_here],
                            marker="*",
                            s=90,
                            facecolor="red",
                            edgecolor="k",
                            linewidth=0.6,
                            zorder=5,
                        )
                else:
                    ax.text(
                        0.5,
                        0.5,
                        "no valid data",
                        ha="center",
                        va="center",
                        transform=ax.transAxes,
                        color="grey",
                        fontsize=7,
                    )
            if row_key == "airmass":
                ax.set_ylim(2.5, 0.8)

            ax.tick_params(axis="both", labelsize=6.5)
            ax.grid(True, lw=0.3, alpha=0.35)
            if col_idx == 0:
                ax.set_ylabel(ylabel, fontsize=7.5)

            if row_idx == n_rows - 1:
                ax.set_xlabel("expMidptMJD", fontsize=8)
            else:
                plt.setp(ax.get_xticklabels(), visible=False)

    plt.tight_layout()
    savefig(fig, file_stem)
    gc.collect()

## 6. Discover per-star files and run the plotting loop

In [ ]:
lc_files = sorted(f for f in os.listdir(DIR_PER_STAR_IN) if f.endswith(PER_STAR_SUFFIX))
log.info("Found %d per-star FGCM-enriched LC files.", len(lc_files))
lc_files

In [ ]:
n_ok = 0
n_err = 0

for fname in lc_files:
    src_path = os.path.join(DIR_PER_STAR_IN, fname)

    try:
        df_star = pd.read_csv(src_path)
    except Exception as exc:
        log.error("  ERROR reading %s: %s", fname, exc)
        n_err += 1
        continue

    simbad_id = (
        df_star["simbad_id"].iloc[0]
        if "simbad_id" in df_star.columns and len(df_star) > 0
        else fname.replace(PER_STAR_SUFFIX, "")
    )

    file_stem = fname.replace(PER_STAR_SUFFIX, "") + "_LC_vs_CALIB"

    log.info("Plotting: %s  (%d rows)", simbad_id, len(df_star))

    try:
        plot_star_calib_comparison(df_star, simbad_id, file_stem)
        n_ok += 1
    except Exception as exc:
        log.error("  ERROR plotting %s: %s", simbad_id, exc)
        plt.close("all")
        n_err += 1

log.info("Done — %d figures saved, %d errors.", n_ok, n_err)

## 7. Quick inline preview of one figure

In [ ]:
# Display the first saved PNG inline for a quick check
from IPython.display import Image, display

saved_pngs = sorted(os.path.join(DIR_FIGS, f) for f in os.listdir(DIR_FIGS) if f.endswith(".png"))

if saved_pngs:
    log.info("Previewing: %s", saved_pngs[0])
    display(Image(filename=saved_pngs[0], width=1400))
else:
    log.warning("No PNG figure found in %s", DIR_FIGS)

## 8. Single-star deep dive

Quick access to one star's figure by `simbad_id`, without re-running the
full loop — useful to iterate on a specific outlier case.


In [ ]:
select_starname = "SDSS J100158.92+021246.1"

target_file = None
for fname in lc_files:
    src_path = os.path.join(DIR_PER_STAR_IN, fname)
    df_tmp = pd.read_csv(src_path, nrows=1)
    if "simbad_id" in df_tmp.columns and df_tmp["simbad_id"].iloc[0] == select_starname:
        target_file = fname
        break

if target_file is None:
    log.warning("Star '%s' not found among per-star files.", select_starname)
else:
    df_star_sel = pd.read_csv(os.path.join(DIR_PER_STAR_IN, target_file))
    file_stem_sel = target_file.replace(PER_STAR_SUFFIX, "") + "_LC_vs_CALIB"
    plot_star_calib_comparison(df_star_sel, select_starname, file_stem_sel)
    plt.show()